# Feature Engineering - BisaKerja

Notebook ini menyiapkan baseline feature engineering untuk job matching, role classification, skill gap analysis, dan dashboard insight. Fokus awalnya adalah validasi input dan guardrail definisi dataset sebelum fitur dibuat.

## Phase 0 - Scope and Input Validation

Tujuan phase ini:

- Memastikan semua dataset processed terbaru dapat dimuat.
- Memvalidasi shape, missing values, duplicate rows, dan kolom wajib.
- Menetapkan use case awal feature engineering.
- Menyimpan konfigurasi global yang akan dipakai phase berikutnya.
- Mencatat bahwa `indotech_job_train.csv` adalah modeling-ready subset, bukan train split final.

In [1]:
from pathlib import Path
import json
from datetime import datetime, timezone

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    """Find repository root from either the project folder or notebooks folder."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "TODOS.md").exists() and (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Project root with TODOS.md and data/processed was not found.")


PROJECT_ROOT = find_project_root()
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURE_DIR = PROJECT_ROOT / "data" / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

INPUT_PATHS = {
    "job_cleaned": DATA_PROCESSED_DIR / "indotech_job_cleaned.csv",
    "job_modeling_ready": DATA_PROCESSED_DIR / "indotech_job_train.csv",
    "resume_cleaned": DATA_PROCESSED_DIR / "techtalent_profile_cleaned.csv",
    "job_nontech_reference": DATA_PROCESSED_DIR / "indotech_nontech.csv",
}

OUTPUT_PATHS = {
    "phase0_validation_summary": FEATURE_DIR / "phase0_input_validation_summary.csv",
    "phase0_config": FEATURE_DIR / "phase0_config.json",
}

IDENTIFIER_COLUMNS = {
    "job": ["job_id"],
    "resume": ["ID"],
}

TARGET_COLUMNS = {
    "job_fit_scoring": [],
    "skill_gap_analysis": [],
    "resume_role_classification": ["Job_Role"],
    "job_tech_classification_or_filtering": ["is_tech"],
}

AUDIT_PIPELINE_COLUMNS = {
    "job": [
        "tech_signal_source",
        "_is_tech_cat",
        "_is_tech_title",
        "_is_tech_skills",
        "is_train_ready",
        "fit_input_quality_score",
        "fit_input_has_requirements",
        "fit_input_has_skills",
    ],
    "resume": [],
}

USE_CASES = [
    "job fit scoring",
    "skill gap analysis",
    "resume/job role classification",
    "dashboard analytics",
]

REQUIRED_COLUMNS = {
    "job_cleaned": [
        "job_id", "title", "normalized_title", "category", "description",
        "requirement_summary", "requirements_concat", "skills_clean",
        "experience_level", "work_type", "employment_type", "province", "city",
        "salary_min", "salary_max", "source_posted_at", "is_tech", "is_train_ready",
    ],
    "job_modeling_ready": [
        "job_id", "title", "normalized_title", "category", "description",
        "requirement_summary", "requirements_concat", "skills_clean",
        "experience_level", "work_type", "employment_type", "province", "city",
        "salary_min", "salary_max", "source_posted_at", "is_tech", "is_train_ready",
    ],
    "resume_cleaned": [
        "ID", "Skills", "Projects", "Education", "Experience", "Job_Role", "Required_Skills",
    ],
    "job_nontech_reference": [
        "job_id", "title", "normalized_title", "category", "description",
        "requirement_summary", "requirements_concat", "skills_top_10_names",
        "experience_level", "work_type", "employment_type", "province", "city",
        "salary_min", "salary_max", "source_posted_at", "is_tech",
    ],
}

config = {
    "random_state": RANDOM_STATE,
    "project_root": str(PROJECT_ROOT),
    "input_paths": {k: str(v.relative_to(PROJECT_ROOT)) for k, v in INPUT_PATHS.items()},
    "output_paths": {k: str(v.relative_to(PROJECT_ROOT)) for k, v in OUTPUT_PATHS.items()},
    "identifier_columns": IDENTIFIER_COLUMNS,
    "target_columns": TARGET_COLUMNS,
    "audit_pipeline_columns": AUDIT_PIPELINE_COLUMNS,
    "use_cases": USE_CASES,
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Feature output directory: {FEATURE_DIR}")

Project root: C:\00 DATA\01 DATA\SEMESTER 6\Metopen\MIKA SMA\BisaKerja\BisaKerja
Feature output directory: C:\00 DATA\01 DATA\SEMESTER 6\Metopen\MIKA SMA\BisaKerja\BisaKerja\data\features


### Load Dataset Utama

In [3]:
datasets = {}
for name, path in INPUT_PATHS.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing input dataset for {name}: {path}")
    datasets[name] = pd.read_csv(path)

[(name, df.shape) for name, df in datasets.items()]

[('job_cleaned', (1679, 35)),
 ('job_modeling_ready', (1678, 35)),
 ('resume_cleaned', (69929, 7)),
 ('job_nontech_reference', (1395, 32))]

### Validasi Shape, Missing Values, Duplicate Rows, dan Kolom Wajib

In [4]:
def summarize_dataset(name: str, df: pd.DataFrame, required_columns: list[str]) -> dict:
    missing_required = sorted(set(required_columns) - set(df.columns))
    duplicate_id_rows = None

    if name.startswith("job") and "job_id" in df.columns:
        duplicate_id_rows = int(df.duplicated(subset=["job_id"]).sum())
    elif name.startswith("resume") and "ID" in df.columns:
        duplicate_id_rows = int(df.duplicated(subset=["ID"]).sum())

    return {
        "dataset": name,
        "path": str(INPUT_PATHS[name].relative_to(PROJECT_ROOT)),
        "rows": len(df),
        "columns": df.shape[1],
        "missing_values": int(df.isna().sum().sum()),
        "missing_value_pct": round(float(df.isna().sum().sum() / (df.shape[0] * df.shape[1]) * 100), 4),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_identifier_rows": duplicate_id_rows,
        "required_columns_count": len(required_columns),
        "missing_required_columns": ", ".join(missing_required) if missing_required else "",
        "required_columns_ok": len(missing_required) == 0,
    }

validation_summary = pd.DataFrame([
    summarize_dataset(name, df, REQUIRED_COLUMNS[name])
    for name, df in datasets.items()
])

validation_summary

,dataset,path,rows,columns,missing_values,missing_value_pct,duplicate_rows,duplicate_identifier_rows,required_columns_count,missing_required_columns,required_columns_ok
0,job_cleaned,data\processed\indotech_job_cleaned.csv,1679,35,0,0.0000,0,0,18,,True
1,job_modeling_ready,data\processed\indotech_job_train.csv,1678,35,0,0.0000,0,0,18,,True
2,resume_cleaned,data\processed\techtalent_profile_cleaned.csv,69929,7,0,0.0000,0,0,7,,True
3,job_nontech_reference,data\processed\indotech_nontech.csv,1395,32,1905,4.2675,0,0,17,,True


In [5]:
missing_by_dataset = []
for name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    for column, count in missing.items():
        missing_by_dataset.append({
            "dataset": name,
            "column": column,
            "missing_values": int(count),
            "missing_pct": round(float(count / len(df) * 100), 2),
        })

missing_by_dataset = pd.DataFrame(missing_by_dataset)
missing_by_dataset if not missing_by_dataset.empty else pd.DataFrame(columns=["dataset", "column", "missing_values", "missing_pct"])

,dataset,column,missing_values,missing_pct
0,job_nontech_reference,salary_min,936,67.10
1,job_nontech_reference,salary_max,931,66.74
2,job_nontech_reference,category,33,2.37
3,job_nontech_reference,province,5,0.36


In [6]:
required_column_check = []
for name, required_cols in REQUIRED_COLUMNS.items():
    available = set(datasets[name].columns)
    for column in required_cols:
        required_column_check.append({
            "dataset": name,
            "column": column,
            "exists": column in available,
        })

required_column_check = pd.DataFrame(required_column_check)
required_column_check.groupby("dataset")["exists"].agg(["sum", "count", "all"]).reset_index()

,dataset,sum,count,all
0,job_cleaned,18,18,True
1,job_modeling_ready,18,18,True
2,job_nontech_reference,17,17,True
3,resume_cleaned,7,7,True


### Catatan Definisi `indotech_job_train.csv`

`indotech_job_train.csv` diperlakukan sebagai **modeling-ready subset** hasil quality gate, bukan train split final. Split final untuk modeling akan dibuat pada Phase 1 agar strategi time-based atau fallback random berbasis `job_id` dapat dikontrol dan metadata split dapat disimpan.

In [7]:
job_cleaned = datasets["job_cleaned"]
job_modeling_ready = datasets["job_modeling_ready"]

cleaned_ids = set(job_cleaned["job_id"])
modeling_ready_ids = set(job_modeling_ready["job_id"])

train_definition_check = pd.DataFrame([
    {
        "check": "job_modeling_ready_is_subset_of_job_cleaned",
        "value": modeling_ready_ids.issubset(cleaned_ids),
    },
    {
        "check": "job_modeling_ready_rows",
        "value": len(job_modeling_ready),
    },
    {
        "check": "job_cleaned_rows",
        "value": len(job_cleaned),
    },
    {
        "check": "rows_in_cleaned_not_in_modeling_ready",
        "value": len(cleaned_ids - modeling_ready_ids),
    },
    {
        "check": "all_modeling_ready_rows_have_is_train_ready_1",
        "value": bool((job_modeling_ready["is_train_ready"] == 1).all()),
    },
])

train_definition_check

,check,value
0,job_modeling_ready_is_subset_of_job_cleaned,True
1,job_modeling_ready_rows,1678
2,job_cleaned_rows,1679
3,rows_in_cleaned_not_in_modeling_ready,1
4,all_modeling_ready_rows_have_is_train_ready_1,True


### Konfigurasi Global Phase 0

In [8]:
config_preview = pd.DataFrame([
    {"config_key": "RANDOM_STATE", "value": RANDOM_STATE},
    {"config_key": "input_paths", "value": json.dumps(config["input_paths"], indent=2)},
    {"config_key": "output_paths", "value": json.dumps(config["output_paths"], indent=2)},
    {"config_key": "identifier_columns", "value": json.dumps(IDENTIFIER_COLUMNS, indent=2)},
    {"config_key": "target_columns", "value": json.dumps(TARGET_COLUMNS, indent=2)},
    {"config_key": "audit_pipeline_columns", "value": json.dumps(AUDIT_PIPELINE_COLUMNS, indent=2)},
    {"config_key": "use_cases", "value": json.dumps(USE_CASES, indent=2)},
])

config_preview

,config_key,value
0,RANDOM_STATE,42
1,input_paths,"{\n ""job_cleaned"": ""data\\processed\\indotech_job_cleaned.csv"",\n ""job_modeling_ready"": ""data\\processed\\indotech..."
2,output_paths,"{\n ""phase0_validation_summary"": ""data\\features\\phase0_input_validation_summary.csv"",\n ""phase0_config"": ""data\\..."
3,identifier_columns,"{\n ""job"": [\n ""job_id""\n ],\n ""resume"": [\n ""ID""\n ]\n}"
4,target_columns,"{\n ""job_fit_scoring"": [],\n ""skill_gap_analysis"": [],\n ""resume_role_classification"": [\n ""Job_Role""\n ],\n ..."
5,audit_pipeline_columns,"{\n ""job"": [\n ""tech_signal_source"",\n ""_is_tech_cat"",\n ""_is_tech_title"",\n ""_is_tech_skills"",\n ""i..."
6,use_cases,"[\n ""job fit scoring"",\n ""skill gap analysis"",\n ""resume/job role classification"",\n ""dashboard analytics""\n]"


### Simpan Artifact Phase 0

In [9]:
validation_summary.to_csv(OUTPUT_PATHS["phase0_validation_summary"], index=False)

config_to_write = {
    **config,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "note": "indotech_job_train.csv is a modeling-ready subset, not the final train split.",
}
with open(OUTPUT_PATHS["phase0_config"], "w", encoding="utf-8") as f:
    json.dump(config_to_write, f, indent=2)

print(f"Saved validation summary: {OUTPUT_PATHS['phase0_validation_summary'].relative_to(PROJECT_ROOT)}")
print(f"Saved phase 0 config: {OUTPUT_PATHS['phase0_config'].relative_to(PROJECT_ROOT)}")

Saved validation summary: data\features\phase0_input_validation_summary.csv
Saved phase 0 config: data\features\phase0_config.json


### Ringkasan Phase 0

- Empat dataset processed berhasil dimuat.
- Kolom wajib untuk baseline feature engineering tersedia.
- Missing values hanya muncul pada `indotech_nontech.csv`, sesuai perannya sebagai reference/negative sample yang tidak melewati quality gate utama.
- Tidak ada duplicate row pada dataset yang divalidasi.
- `indotech_job_train.csv` dikunci definisinya sebagai modeling-ready subset, bukan train split final.

## Phase 1 - A/B Testing Feature Preparation

Bagian ini membuat rancangan eksperimen A/B untuk fitur rekomendasi BisaKerja.

- **Variant A:** baseline recommendation memakai overlap kandidat terhadap demand skill entry-level global.
- **Variant B:** personalized recommendation memakai kombinasi demand skill entry-level dan role-aware skill gap.
- **Unit randomisasi:** kandidat (`ID`).
- **Offline proxy metric:** `observed_fit_score_proxy` dan `is_high_fit_proxy`. Metric produksi yang disarankan tetap memakai event nyata seperti `apply_click_rate`, `profile_completion_rate`, atau `application_submit_rate`.


In [10]:
import re
import numpy as np

AB_OUTPUT_PATHS = {
    'ab_candidate_assignments': FEATURE_DIR / 'ab_testing_candidate_assignments.csv',
    'ab_summary': FEATURE_DIR / 'ab_testing_summary.csv',
    'ab_config': FEATURE_DIR / 'phase1_ab_testing_config.json',
}

skill_tax_path = PROJECT_ROOT / 'data' / 'external' / 'skill_taxonomy_bisakerja_v3.csv'
skill_tax = pd.read_csv(skill_tax_path)
skill_alias = (
    skill_tax[['skill_raw', 'skill_standard']]
    .dropna()
    .assign(
        skill_raw=lambda d: d['skill_raw'].str.strip().str.lower(),
        skill_standard=lambda d: d['skill_standard'].str.strip().str.lower(),
    )
    .drop_duplicates('skill_raw')
    .set_index('skill_raw')['skill_standard']
    .to_dict()
)

ARTIFACT_SKILL_PATTERNS = [r'expire', r'berakhir', r'pelamar', r'early_applicant']

def split_and_normalize_skills(text):
    if pd.isna(text):
        return []
    raw_items = str(text).replace('|', ',').split(',')
    clean_items = []
    for item in raw_items:
        skill = item.strip().lower()
        if not skill:
            continue
        skill = skill_alias.get(skill, skill)
        if any(re.search(pattern, skill, flags=re.IGNORECASE) for pattern in ARTIFACT_SKILL_PATTERNS):
            continue
        clean_items.append(skill)
    return sorted(set(clean_items))

def overlap_ratio(candidate_skills, reference_skills):
    candidate_set = set(candidate_skills)
    reference_set = set(reference_skills)
    if not reference_set:
        return 0.0
    return len(candidate_set.intersection(reference_set)) / len(reference_set)

jobs_ab = datasets['job_modeling_ready'].copy()
resume_ab = datasets['resume_cleaned'].copy()

jobs_ab['skill_list'] = jobs_ab['skills_clean'].apply(split_and_normalize_skills)
resume_ab['skill_list'] = resume_ab['Skills'].apply(split_and_normalize_skills)
resume_ab['required_skill_list'] = resume_ab['Required_Skills'].apply(split_and_normalize_skills)

entry_job_skills = (
    jobs_ab.loc[jobs_ab['experience_level'].eq('ENTRY_LEVEL'), 'skill_list']
    .explode()
    .dropna()
    .value_counts()
)
entry_market_skill_ref = entry_job_skills.head(20).index.tolist()

ab_candidates = (
    resume_ab[resume_ab['Experience'].isin(['Fresher', '1-2 years'])]
    .sort_values('ID')
    .reset_index(drop=True)
    .copy()
)

rng = np.random.default_rng(RANDOM_STATE)
ab_candidates['ab_variant'] = np.where(rng.random(len(ab_candidates)) < 0.5, 'A', 'B')

ab_candidates['baseline_entry_score'] = ab_candidates['skill_list'].apply(
    lambda skills: overlap_ratio(skills, entry_market_skill_ref) * 100
)
ab_candidates['role_aware_score'] = ab_candidates.apply(
    lambda row: overlap_ratio(row['skill_list'], row['required_skill_list']) * 100,
    axis=1,
)
ab_candidates['variant_b_score'] = (
    0.65 * ab_candidates['baseline_entry_score']
    + 0.35 * ab_candidates['role_aware_score']
)
ab_candidates['observed_fit_score_proxy'] = np.where(
    ab_candidates['ab_variant'].eq('A'),
    ab_candidates['baseline_entry_score'],
    ab_candidates['variant_b_score'],
)

HIGH_FIT_THRESHOLD = 25
ab_candidates['is_high_fit_proxy'] = ab_candidates['observed_fit_score_proxy'] >= HIGH_FIT_THRESHOLD
ab_candidates['skill_count'] = ab_candidates['skill_list'].apply(len)
ab_candidates['required_skill_count'] = ab_candidates['required_skill_list'].apply(len)
ab_candidates['missing_required_skill_count'] = ab_candidates.apply(
    lambda row: len(set(row['required_skill_list']) - set(row['skill_list'])),
    axis=1,
)

ab_candidate_assignments = ab_candidates[[
    'ID', 'Experience', 'Job_Role', 'ab_variant',
    'skill_count', 'required_skill_count', 'missing_required_skill_count',
    'baseline_entry_score', 'role_aware_score', 'variant_b_score',
    'observed_fit_score_proxy', 'is_high_fit_proxy',
]].copy()

ab_candidate_assignments.head()


,ID,Experience,Job_Role,ab_variant,skill_count,required_skill_count,missing_required_skill_count,baseline_entry_score,role_aware_score,variant_b_score,observed_fit_score_proxy,is_high_fit_proxy
0,1,Fresher,Web Developer,B,8,5,2,5.0,60.0,24.25,24.25,False
1,2,Fresher,ML Engineer,A,8,5,2,5.0,60.0,24.25,5.00,False
2,4,1-2 years,Data Scientist,B,8,5,1,10.0,80.0,34.50,34.50,True
3,6,Fresher,Web Developer,B,8,5,2,10.0,60.0,27.50,27.50,True
4,7,1-2 years,Cybersecurity Analyst,A,8,5,2,5.0,60.0,24.25,5.00,False


In [11]:
ab_balance_summary = ab_candidate_assignments.groupby('ab_variant').agg(
    candidates=('ID', 'count'),
    avg_skill_count=('skill_count', 'mean'),
    avg_missing_required_skill_count=('missing_required_skill_count', 'mean'),
    avg_observed_fit_score_proxy=('observed_fit_score_proxy', 'mean'),
    high_fit_proxy_rate=('is_high_fit_proxy', 'mean'),
).reset_index()

ab_balance_summary['high_fit_proxy_rate'] = (ab_balance_summary['high_fit_proxy_rate'] * 100).round(2)
ab_balance_summary[['avg_skill_count', 'avg_missing_required_skill_count', 'avg_observed_fit_score_proxy']] = (
    ab_balance_summary[['avg_skill_count', 'avg_missing_required_skill_count', 'avg_observed_fit_score_proxy']]
    .round(2)
)

role_balance = pd.crosstab(
    ab_candidate_assignments['Job_Role'],
    ab_candidate_assignments['ab_variant'],
    normalize='index',
).mul(100).round(2)

print('Balance summary:')
display(ab_balance_summary)
print('Role balance (% per role):')
display(role_balance)


Balance summary:


,ab_variant,candidates,avg_skill_count,avg_missing_required_skill_count,avg_observed_fit_score_proxy,high_fit_proxy_rate
0,A,17380,7.66,1.77,8.11,0.25
1,B,17535,7.66,1.77,27.95,62.00


Role balance (% per role):


ab_variant,A,B
Job_Role,,
Backend Developer,49.19,50.81
Business Analyst,49.76,50.24
Cloud Engineer,50.58,49.42
Cybersecurity Analyst,50.73,49.27
Data Scientist,50.04,49.96
ML Engineer,49.31,50.69
Web Developer,48.79,51.21


In [12]:
ab_candidate_assignments.to_csv(AB_OUTPUT_PATHS['ab_candidate_assignments'], index=False)
ab_balance_summary.to_csv(AB_OUTPUT_PATHS['ab_summary'], index=False)

ab_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'random_state': RANDOM_STATE,
    'experiment_name': 'early_career_recommendation_ab_test',
    'unit_of_randomization': 'candidate_id',
    'variant_a': 'baseline entry-level market skill overlap',
    'variant_b': 'personalized role-aware skill-gap recommendation',
    'entry_market_skill_ref_top_n': 20,
    'entry_market_skill_ref': entry_market_skill_ref,
    'primary_offline_proxy_metric': 'observed_fit_score_proxy',
    'secondary_offline_proxy_metric': 'is_high_fit_proxy',
    'high_fit_threshold': HIGH_FIT_THRESHOLD,
    'production_metrics_recommended': [
        'apply_click_rate',
        'application_start_rate',
        'application_submit_rate',
        'profile_completion_rate',
    ],
    'output_paths': {k: str(v.relative_to(PROJECT_ROOT)) for k, v in AB_OUTPUT_PATHS.items()},
    'important_note': 'Offline proxy metrics are not a substitute for real randomized production event logs.',
}

with open(AB_OUTPUT_PATHS['ab_config'], 'w', encoding='utf-8') as f:
    json.dump(ab_config, f, indent=2)

print(f"Saved assignments: {AB_OUTPUT_PATHS['ab_candidate_assignments'].relative_to(PROJECT_ROOT)}")
print(f"Saved summary: {AB_OUTPUT_PATHS['ab_summary'].relative_to(PROJECT_ROOT)}")
print(f"Saved config: {AB_OUTPUT_PATHS['ab_config'].relative_to(PROJECT_ROOT)}")


Saved assignments: data\features\ab_testing_candidate_assignments.csv
Saved summary: data\features\ab_testing_summary.csv
Saved config: data\features\phase1_ab_testing_config.json


**Interpretasi awal:** random assignment ini dipakai sebagai kerangka eksperimen dan simulasi offline. Kesimpulan bisnis final baru valid setelah Variant A dan Variant B benar-benar ditampilkan ke pengguna, lalu event produksi dianalisis dengan metrik yang sama untuk kedua grup.
